# Lab 05 · ReAct agents: reason, act, observe

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Warm-up · Think, act, observe, then answer

ReAct stands for **reasoning and acting**. In this small loop the model gives a short reason and chooses `LOOKUP` or `FINISH`. `LOOKUP` reads the sample weather stored in the notebook. Python gives the model that result, called an **observation**, on the next turn. If the model requests an unknown action or tries to finish early, Python sends the error back on the next turn. After a successful lookup, `FINISH` lets the model suggest an activity based on the returned weather. You have three turns to see this happen.

In [ ]:
weather = None
feedback = ""
for turn in range(3):
    reply = ask(f"Known weather: {weather}. {feedback}\n"
                "Reply in two lines: line 1, one short reason. Line 2, exactly LOOKUP or FINISH. "
                "If weather is unknown, LOOKUP before FINISH.")
    lines = [line.strip() for line in reply.strip().splitlines() if line.strip()]
    thought = " ".join(lines[:-1]) or "(no reason given)"
    action = (lines[-1] if lines else "").upper().strip(".")
    print("Turn", turn + 1, "· thought:", thought, "| action:", action)
    if action == "LOOKUP":
        weather = "rainy"  # Result of a local teaching tool.
        feedback = ""
        print("Observation:", weather)
    elif action == "FINISH" and weather is not None:
        print("Advice:", ask(f"Weather: {weather}. Suggest one indoor or outdoor activity."))
        break
    else:
        if action == "FINISH":
            feedback = "LOOKUP is required before FINISH."
        else:
            feedback = "Unknown action. Choose LOOKUP or FINISH."
        print("Error observation:", feedback)
else:
    print("Loop ended without an accepted FINISH. Rerun the cell.")

**Next:** Apply the same think–act–observe loop to simulated dependency metrics.

## Inspect a dependency before choosing its resilience policy
A **dependency** is a service that checkout relies on, such as Inventory. When Inventory is slow, services waiting for it can run out of capacity and become slow themselves. To choose a response, first look at the symptoms and ask what happens if a request is repeated.

In Lab 04 you let a model request functions and see their results. Here you make the **ReAct** cycle explicit. The model gives a short reason and requests an action. Python runs it and returns an **observation**, the result the model can use on its next turn. The model decides which action to try within limits set by the notebook.

The exercise uses three policy labels.

- **Circuit breaker** temporarily stops calls to a dependency that is failing frequently. The exercise chooses it at a failure rate of 50% or more.
- **Bounded retry** tries again a limited number of times. Below 50%, the exercise chooses it for a temporary failure on an idempotent operation, one that has the same intended effect when repeated.
- **No automatic retry** means the caller should avoid retrying on its own. For example, repeating a payment can charge the customer twice.

The services, numbers, and thresholds are teaching examples. The agent recommends a label, and Python checks it against the exercise policy.

**At a glance**

![Lab 05 at a glance: Goal, then Thought + action (one-line reason · JSON), then Guards (known tool · facts read), then Run the tool (read target metrics), then Observation (added to history). Then back to Thought + action: repeat · at most 5 turns.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/05.svg)

## Think about the example

A slow dependency can occupy callers while they wait. Retrying adds more requests. A repeated card charge can have a second financial effect when the operation is not idempotent.

**Concepts:** Cascading failure; Circuit breaker vs retry; Idempotency and graceful degradation.

Use the scenario above to consider these questions before running the code:

1. Why can retries make a slow dependency worse?
2. Why must the non-idempotent Payments charge not be blindly retried?
3. What does printing the model's one-line thought add over an action alone?

## The idea in context

ReAct-style control flow alternates a short model thought, a tool action, and the observation it returns. The example exposes those exchanges without deploying a checkout system.

An agent loop thinks, then selects its next step from observations; it still needs a bound and allowed tools.

| Key term | Meaning |
| --- | --- |
| **Thought** | The model's one-line reason for choosing its next action. |
| **Agent** | A system in which a model chooses its next action in a loop, based on what it has observed. |
| **Observation** | The result of an action, fed back into the next prompt. |
| **History** | Every thought, action and observation so far; the agent's view of the task. |
| **Guard** | A rule in code that the agent cannot override, such as a turn limit. |

## Pattern · One-Step Tool Agent in a ReAct loop

*Related to the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Connection** | Lab 04 used a small tool loop to look up, validate, and format data. Here each turn asks for a short explanation and one action. The next turn receives the observation from that action. |
| **Flow** | Goal → Thought → Action → Tool → Observation → repeat, or Finish |
| **Guard** | A loop can run forever or finish without reading metrics for the target. The turn limit and finish guard address both. |

## Main exercise at a glance

1. Prepare observations
2. Define the task, the policy, and the actions
3. Think, act, observe, and repeat
4. Check the recommendation and its limits

## Step 1 · Prepare observations
The `METRICS` dictionary holds two sample service records. A `failure_rate` of `0.7` means 70% of requests fail in the example. `peak_ms` represents the highest latency in the example, and `p99_ms` represents the latency at or below which about 99% of requests finish. This cell prints the dependency names for you. In the following steps, the model has to ask to see a record.

In [ ]:
import json

METRICS = {
    "Inventory": {"service": "Inventory", "failure_rate": 0.7, "peak_ms": 5000,
                  "idempotent": True, "transient_failure": True},
    "Payments": {"service": "Payments", "failure_rate": 0.01, "p99_ms": 1500,
                 "idempotent": False, "transient_failure": True},
}
print("Available dependencies:", list(METRICS))

## Step 2 · Define the task, the policy, and the actions
`target` is the service to inspect. The model can ask for `list_dependencies` to get the exact names, `read_metrics` with one name, or `finish` with a policy label. The prompt requests a one-line reason in the `thought` field. This is an explanation supplied for the exercise.

`run_tool` uses ordinary `if` statements to execute an allowed action and return an observation. It returns an error observation for an unknown action or service. `POLICY` tells the model the exercise rules in words, and `expected_policy` expresses those rules separately in Python for the final check.

In [ ]:
target = "Inventory"
POLICY = (
    "For this exercise: failure_rate >= 0.5 -> circuit breaker; "
    "otherwise idempotent AND transient_failure -> bounded retry; otherwise no automatic retry. "
    "Never infer metrics: read the target's record first."
)
FORMAT = 'JSON only: {"thought":"one short reason","action":"tool name or finish","input":"service name or policy label"}'

def run_tool(action, value):
    """Execute an allowed action and return its observation."""
    if action == "list_dependencies":
        return list(METRICS)
    if action == "read_metrics":
        return METRICS.get(value, {"error": "Unknown dependency; list exact names first."})
    return {"error": "Unknown action; use an allowed tool."}

def expected_policy(facts):
    """The exercise policy as code, used by the checks in Step 4 and CrewAI."""
    if facts["failure_rate"] >= 0.5:
        return "circuit breaker"
    if facts["idempotent"] and facts["transient_failure"]:
        return "bounded retry"
    return "no automatic retry"

print("Target:", target)

## Step 3 · Think, act, observe, and repeat
Each turn sends the goal, policy, and `history`, a list of earlier actions and observations that Python stores and resends. The model replies with JSON and Python acts on the parsed `action` and `input`. For example, a request with action `read_metrics` and input `Inventory` adds the Inventory record to `history`.

The **finish guard** is a rule in Python. It checks that a successful lookup returned the target's metrics. A malformed reply, an unknown service, or a request to finish too early produces an error observation that the next turn can use. Each attempt counts toward the five-turn limit.

In [ ]:
history, decision, observed = [], None, False
for turn in range(1, 6):
    prompt = (f"{POLICY}\nGoal: recommend a resilience policy for {target}.\n"
              "Tools: list_dependencies (input ignored), read_metrics (exact service name).\n"
              f"History: {json.dumps(history)}\n{FORMAT}")
    try:
        step = json.loads(ask(prompt))
    except json.JSONDecodeError:
        step = None
    if not isinstance(step, dict) or not isinstance(step.get("action"), str) or not isinstance(step.get("input"), str):
        observation = {"error": "Reply with one JSON object holding thought, action and input."}
        history.append({"action": "invalid", "input": "", "observation": observation})
        print("Turn", turn, "· rejected:", observation["error"])
        continue
    action, value = step["action"].strip().lower(), step["input"].strip().rstrip(".")
    print(f"Turn {turn} · thought: {step.get('thought', '(none)')} | action: {action} {value}")
    if action == "finish":
        if not observed:
            observation = {"error": "Read the target's metrics successfully before finishing."}
            history.append({"thought": step.get("thought", ""), "action": action, "input": value, "observation": observation})
            print("  guard:", observation["error"])
            continue
        decision = value.lower()
        break
    observation = run_tool(action, value)
    if action == "read_metrics" and isinstance(observation, dict) and observation.get("service") == target and "failure_rate" in observation:
        observed = True
    history.append({"thought": step.get("thought", ""), "action": action, "input": value, "observation": observation})
    print("  observation:", observation)
assert decision is not None, "Turn budget exhausted; escalate instead of guessing."

## Step 4 · Check the recommendation and its limits
`expected_policy` reads the target's simulated record from `METRICS` and calculates the expected label in Python. Compare `Recommendation` with `Exercise expects`. The default Inventory record has a 70% failure rate, so the check expects `circuit breaker`. The check verifies the recommendation against the rules of this exercise.

In [ ]:
expected = expected_policy(METRICS[target])
print("Recommendation:", decision)
print("Exercise expects:", expected, "| observations:", len(history))
assert decision == expected, "Recommendation fails the exercise policy; rerun Step 3 and read the trace."
print("Blindly retrying Payments can duplicate its non-idempotent charge.")

## What you should see
A successful run prints a `read_metrics Inventory` action, an observation with Inventory's record, and the checked recommendation `circuit breaker`. It may list dependencies first. An error observation gives the model a chance to correct its request within the five-turn limit.

If you see `Turn budget exhausted`, read the printed actions and check whether Inventory was ever read successfully. Rerun Step 3 for a fresh attempt. If Step 4 rejects the recommendation, rerun Step 3 and then Step 4 so the check sees the latest recommendation.

## Takeaway
The model uses each observation to choose its next action. Python controls the available actions, how many times the model can try, when it may finish, and how its answer is checked.

## Optional · The same ReAct example with CrewAI

In Step 3, you wrote the loop yourself. **CrewAI** is a framework that can manage the repeated model and tool calls. This example gives one agent the same target, policy, and a function that reads the same simulated metrics.

Run Setup and Steps 1–2 before running this optional section. Start with the install cell. If Colab requests a restart, rerun Setup and Steps 1–2, then continue below. This example uses your selected provider's Colab secret and a separate client. `max_iter` limits the agent's attempts, and the number of model requests may differ. The manual exercise already demonstrates the full cycle.

In [ ]:
%pip install -q "crewai==1.15.45" "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"

### 1 · Give the agent a metrics tool

The decorator turns the Python function into a CrewAI tool. Its description tells the agent how to use it. Successful lookups are recorded so we can verify the agent actually read the target. The data is simulated; no real service is queried.

In [ ]:
from crewai.tools import tool
import json

observed_services = []

@tool("read_metrics")
def read_metrics(service: str) -> str:
    """Read simulated metrics for the exact service name: Inventory or Payments."""
    record = METRICS.get(service)
    if record is None:
        return "Unknown dependency. Use Inventory or Payments."
    observed_services.append(service)
    print("Observation:", record)
    return json.dumps(record)

### 2 · Create one agent and one task

CrewAI uses the chosen provider’s OpenAI-compatible endpoint. The `openai/` prefix selects CrewAI’s compatible client; it does not change the provider or send the key to OpenAI. The agent can call only the supplied tool. A small iteration limit bounds its decision loop.

In [ ]:
from crewai import Agent, Crew, LLM, Task
from coursekit import PROVIDERS, secret

settings = PROVIDERS[PROVIDER]
llm = LLM(
    model="openai/" + settings["model"], custom_openai=True,
    base_url=settings["base_url"], api_key=secret(settings["secret_name"]),
    stream=False, max_tokens=1024, extra_body=settings.get("extra_body"),
)
agent = Agent(
    role="Resilience reviewer", goal="Recommend a safe resilience policy",
    backstory="You inspect a service before recommending a response.",
    llm=llm, tools=[read_metrics], max_iter=5,
    allow_delegation=False, cache=False, verbose=True,
)
task = Task(
    description=f"Read metrics for {target}. {POLICY} "
                "Return only the policy label, without explanation.",
    expected_output="One label: circuit breaker, bounded retry, or no automatic retry.",
    agent=agent,
)
crew = Crew(agents=[agent], tasks=[task], verbose=True)

### 3 · Run the crew and check the result

Look at the printed tool calls and observations. After CrewAI finishes, Python checks that the target was read and that the recommendation matches the exercise policy. That check runs after the answer, while the manual loop checks whether the model is allowed to finish before it accepts the answer. Rerun this cell to start a fresh observation record.

Use Colab or a notebook that supports `await` at the top level. The `await` expression waits for the crew's asynchronous run to return a result.

In [ ]:
observed_services.clear()
result = await crew.kickoff_async()
answer = result.raw.strip().lower().rstrip(".")
print("CrewAI recommendation:", answer)
print("Services read:", observed_services)

if target not in observed_services:
    raise ValueError("The agent did not read the target metrics; do not accept its answer.")
expected = expected_policy(METRICS[target])
print("Exercise expects:", expected)
assert answer == expected, "The recommendation does not match the exercise policy."

Compare the trace with the manual loop in Step 3. Look for the requested service, the returned metrics, and the final label. To try another record, set `target` to `"Payments"` in Step 2 and rerun Step 2 and the CrewAI cells.

## Exercise · Change the dependency and predict the policy

1. In Step 2, change `target = "Inventory"` to `target = "Payments"`. In Step 1, find the Payments record and predict the policy from its failure rate and `idempotent` field.
2. Rerun Steps 2–4. Find the action that reads the Payments metrics and the printed `Recommendation` and `Exercise expects` labels.
3. Explain why Step 4 expects `no automatic retry` for Payments. Use the observed record to explain what could happen if a timed-out charge were repeated.

**More to try**

- In Step 3's prompt, ask the agent to use `read_metrics` on `stock` first. Check whether the model follows the instruction and whether that error unlocks `finish`.
- Restore `target = "Inventory"`. In Step 1, change Inventory's `failure_rate` from `0.7` to `0.1`. Rerun Steps 1–4 and explain why the expected policy changes to `bounded retry`.
- Pick a timeout, breaker, and fallback for a slow Recommendations dependency. Which customer-facing feature would degrade?